# Import Required Packages

In [1]:
# Basic Packages
import os
import sys
import warnings
import pickle
warnings.filterwarnings("ignore")
from typing import List


# Standard ML Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin


# Tensorflow packages
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, 
                                    Dense)
from tensorflow.keras.optimizers import (Adam,
                                         AdamW)
from tensorflow.keras.losses import (CategoricalCrossentropy, 
                                     SparseCategoricalCrossentropy)

from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint,
                                        TensorBoard)

# Load the Training Samples

In [2]:
with open("../artifacts/trainingData/training_ready_samples.pkl", "rb") as f:
    training_samples = pickle.load(f)
    print("Done!! Loading the Training Samples")

Done!! Loading the Training Samples


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(training_samples[0], training_samples[1], train_size = 0.9)
print(f"Shape of X_train : {X_train.shape}")
print(f"Shape of X_val : {X_val.shape}")
print(f"Shape of y_train : {y_train.shape}")
print(f"Shape of y_val : {y_val.shape}")


Shape of X_train : (6705, 4)
Shape of X_val : (745, 4)
Shape of y_train : (6705, 1)
Shape of y_val : (745, 1)


In [4]:
training_generator = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(64)
val_generator = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(64)


2025-07-24 13:48:38.784868: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-07-24 13:48:38.784893: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-07-24 13:48:38.784979: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
I0000 00:00:1753382918.787965 4786233 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1753382918.788006 4786233 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


# Keras Tuner Neural Network

#### Step 1: Neural Network Architecture

In [5]:
class CustomEmbeddingsModel(Model):

    def __init__(self, hp, num_classes):
        super(CustomEmbeddingsModel, self).__init__()

        # Tunable Parameter for layer 1
        dense_units_1 = hp.Int("units", min_value = 256, max_value = 2058, step = 128)

        # Dense layers
        self.dense_layer_1 = Dense(units = dense_units_1, 
                                 activation = "leaky_relu",
                                 )

        # Dense layers
        self.dense_layer_2 = Dense(units = 128, 
                                 activation = "leaky_relu")
        self.dense_layer_3 = Dense(units = 128, 
                                   activation = "leaky_relu")
        self.output_layer = Dense(units = num_classes, 
                                   activation = "softmax") 


    def call(self, inputs):

        # Dense layer 
        x = self.dense_layer_1(inputs)
        x = self.dense_layer_2(x)
        x = self.dense_layer_3(x)

        # Output layer
        out = self.output_layer(x)

        return out       


#### Step 2 : Keras Tuner Hyperband Model

In [6]:
def keras_tuner_hyperband_model(hp):

    model = CustomEmbeddingsModel(hp, num_classes=2006)

    # Compile the model 
    hp_optimizers = hp.Choice("optimizers", values = ["Adam", "AdamW"])
    model.compile(optimizer = hp_optimizers, loss = SparseCategoricalCrossentropy(),
                  metrics = ['accuracy','f1_score'])
    
    return model

#### Step 3: Instantiate the tuner and perform hypertuning

In [7]:
tuner = kt.Hyperband(keras_tuner_hyperband_model,
                     objective = "val_accuracy", 
                     max_epochs = 3, 
                     directory = "../model/",
                     project_name = "cbow_tuner")


#### Step 4: Early Stopping

In [8]:
early_stopping_callback = EarlyStopping(
    verbose = 0, 
    mode = "min",
    patience = 10
)

callbacks = [early_stopping_callback]

# Step 5: Tuner Search

In [9]:
tuner.search(training_generator, 
             epochs = 100, 
             validation_data = val_generator,
             callbacks = callbacks)


# Best Hyperparametersm
best_hps = tuner.get_best_hyperparameters()[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first LSTM layer is {best_hps.get("units")} and the best choice of optimizer is
{best_hps.get("optimizers")}
""")

Trial 6 Complete [00h 00m 05s]
val_accuracy: 0.038926173001527786

Best val_accuracy So Far: 0.048322148621082306
Total elapsed time: 00h 00m 19s

The hyperparameter search is complete. The optimal number of units in the first LSTM layer is 1024 and the best choice of optimizer is
Adam



In [12]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(training_generator, validation_data = val_generator, epochs=50, callbacks=callbacks)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.0171 - f1_score: 9.9695e-04 - loss: 15.2131 - val_accuracy: 0.0295 - val_f1_score: 9.9656e-04 - val_loss: 7.3006
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0347 - f1_score: 9.9685e-04 - loss: 6.4934 - val_accuracy: 0.0282 - val_f1_score: 9.9680e-04 - val_loss: 7.7771
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0302 - f1_score: 9.9687e-04 - loss: 6.4519 - val_accuracy: 0.0188 - val_f1_score: 9.9685e-04 - val_loss: 8.2855
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0318 - f1_score: 9.9686e-04 - loss: 6.4142 - val_accuracy: 0.0376 - val_f1_score: 9.9655e-04 - val_loss: 8.5348
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0348 - f1_score: 9.9684e-04 - loss: 6.2653 - val_accuracy: 0.0349 - val_f1_score: 9.9684e-04 - val_loss: 8.7052
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0382 - f1_score: 9.9683e-04 - loss: 6